# Iberdrola Datathon: EV Infrastructure Roadmap

**Team:** Colomany

## General Description
This notebook provides a detailed end-to-end pipeline for the **Iberdrola Datathon**. We transform raw geospatial road data into a discrete point-based network enriched with traffic intensity, electrical capacity, and proximity to existing infrastructure. Finally, we solve a grid-aware optimization model to identify the best locations for new ultra-fast EV chargers.

## Table of Contents
1. [Phase 0: Environment Setup & Data Acquisition](#phase-0)
2. [Phase 1: Demand Forecasting (EV Registrations)](#phase-1)
3. [Phase 2: Spatial Backbone Foundation](#phase-2)
4. [Phase 3: Grid-Aware Optimization](#phase-3)
5. [Phase 4: Results & Visualization](#phase-4)

---


# Phase 0: Environment Setup & Data Acquisition <a name="phase-0"></a>


### 0.1 Repository Setup
**Procedure**: Cloning the project repository to the local environment. Important for Google Colab sessions.

In [ ]:
!git clone https://github.com/JJR9903/Iberdrola-Datathon.git

### 0.2 Working Directory
**Procedure**: Changing the current working directory to the project root. Important for Google Colab sessions.


In [ ]:
cd "/content/Iberdrola-Datathon"

### 0.3 Library & Script Imports
**Procedure**: Importing essential libraries (Pandas, GeoPandas, etc.) and custom processing functions.
**Details**:
- `discretize_backbone_roads`: To convert LineStrings to Points.
- `map_traffic_to_points`: To project traffic data.
- `assign_nearest_charging_stations`/`gas_stations`: Proximity analysis.
**Main Assumptions**: All dependencies listed in `pyproject.toml` are installed. (For Google Colab sessions, these dependencies are usually installed by default)


In [ ]:
import geopandas as gpd
import pandas as pd
import folium
import os
import numpy as np
import tomllib
import sys
import polars as pl
import statsmodels.api as sm
import pmdarima as pm
from datetime import date
from dateutil.relativedelta import relativedelta



# Change directory to root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add scripts directory to path to allow module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data
from scripts.processing.create_backbone_foundation import (
    discretize_backbone_roads,
    map_traffic_to_points,
    assign_nearest_charging_stations,
    assign_nearest_gas_stations,
    assign_grid_capacity
)


### 0.4 Cloud Data Synchronization

> **Note**: A previous data extraction and standardization process was performed using the project's specialized scripts (scripts/01_acquisition.py for downloading the raw data files, bronze layer) and (scripts/02_standardization.py to standardize each dataset to some specific columns and save them in a parquet format that saves space in memory, Silver layers). For reproducibility and efficiency, this notebook starts by synchronizing these processed **Silver Data** files directly from the cloud to ensure a consistent baseline for analysis.

**Procedure**: Using `sync_standardized_data()` to ensure all required Parquet files are present in the local `data/` directory.
**Datasets Used**: Downloads `roads.parquet`, `traffic.parquet`, `chargers.parquet`, `electric_capacity.parquet`, `vehicle_registrations.parquet`, `gas_stations.parquet`.
**Main Assumptions**: 
- *Assumption*: Cloud storage is accessible.
- *Rationale*: This ensures the "Silver Layer" (standardized data) is available for analysis.


In [ ]:
# 1. Sync data (Ensures silver layer is present)
sync_standardized_data()


### 0.5 Configuration Loading
**Procedure**: Loading parameters from `config.toml` for the backbone foundation step.
**Datasets Used**: `config.toml`.
**Main Assumptions**: The config file contains a `backbone_foundation` section with valid paths and parameters.


In [ ]:
# 2. Load Config
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
params = config['steps']['backbone_foundation']


### 0.6 Data Loading
**Procedure**: Reading the standardized Parquet files into GeoDataFrames and Polars DataFrames.
**Datasets Used**:
- `gdf_roads`: Road backbone segments.
- `gdf_traffic`: Traffic intensity data.
- `gdf_chargers`: Existing EV chargers.
- `gdf_gas`: Traditional gas stations.
- `gdf_capacity`: Electric grid substation capacity.
- `pl_registrations`: Historical vehicle registrations.
**Main Assumptions**: File paths in `config.toml` are correct.


In [ ]:
# 3. Load Standardized Data

print("\n🚀 Loading Standardized Data...")
gdf_roads = gpd.read_parquet(params['roads_path'])
gdf_traffic = gpd.read_parquet(params['traffic_path'])
gdf_chargers = gpd.read_parquet(params['chargers_path'])
gdf_gas = gpd.read_parquet(params['gas_stations_path'])
gdf_capacity = gpd.read_parquet(params['capacity_path'])
pl_registrations = pl.read_parquet('data/standardized/vehicle_registrations.parquet')

print(f" - Roads: {len(gdf_roads)} segments")
print(f" - Traffic: {len(gdf_traffic)} segments")
print(f" - Chargers: {len(gdf_chargers)} sites")
print(f" - Gas Stations: {len(gdf_gas)} sites")
print(f" - Electric Capacity: {len(gdf_capacity)} sites")
print(f" - Vehicle Registrations: {len(pl_registrations)}")

print("\n✅ Data loaded correctly.")

**Procedure**: Verifying the current working directory.


In [ ]:
os.getcwd()

**Procedure**: Visualizing the first few rows of the roads dataset.


In [ ]:
gdf_roads.head()

**Procedure**: Visualizing the traffic dataset segments.


In [ ]:
gdf_traffic.head()

**Procedure**: Visualizing existing charging stations.


In [ ]:
gdf_chargers.head()

**Procedure**: Visualizing traditional gas stations.


In [ ]:
gdf_gas.head()

**Procedure**: Visualizing raw registration data.


In [ ]:
pl_registrations.head()

**Procedure**: Visualizing electric grid capacity (substations).


In [ ]:
gdf_capacity.head()

# Phase 1: Demand Forecasting (EV Registrations) <a name="phase-1"></a>

In this phase, we analyze historical vehicle registration data from DGT to forecast the penetration of Electric Vehicles (EV) in Spain by 2027.

**Methodology Reference**:
- Inspired by the [Laboratorio de Datos GitHub Repo](https://github.com/Admindatosgobes/Laboratorio-de-Datos/tree/main/Data%20Science/Ruta%20a%20la%20electrificaci%C3%B3n%20de%20la%20Movilidad)
- and the [Ruta a la electrificación Colab Notebook](https://colab.research.google.com/github/Admindatosgobes/Laboratorio-de-Datos/blob/main/Data%20Science/Ruta%20a%20la%20electrificaci%C3%B3n%20de%20la%20Movilidad/Codigo/Notebook.ipynb#scrollTo=0b9269b0-d9a3-46d1-8265-ed8a17f086ff)


### 1.1 Data Filtering
**Procedure**: Filtering the registrations to focus on the 2015-2025 period.
**Datasets Used**: `pl_registrations`.
**Main Assumptions**: 
- *Assumption*: 2015 is a sufficient starting point for EV trends.
- *Rationale*: Older data may not reflect modern adoption rates.


In [ ]:
pl_registrations = pl_registrations.filter(
   ( pl.col('date') >= date(2015, 1, 1) ) & (pl.col('date') < date(2026, 1, 1)) 
)

In [ ]:
non_ev_registrations = pl_registrations.filter(pl.col('propulsion') != 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')    
).sort("date").to_pandas().set_index('date')

ev_registrations = pl_registrations.filter(pl.col('propulsion') == 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date").to_pandas().set_index('date')


### 1.2 EV Demand Forecast
**Procedure**: Using **Auto-ARIMA** to find optimal parameters and fitting a **SARIMAX** model to the **log of EV registrations**.
**Datasets Used**: `ev_registrations`.
**Main Assumptions**:
- *Assumption*: Log-transformation handles exponential growth better.
- *Rationale*: Registration trends often show multiplicative seasonality.
- *Assumption*: 12-month seasonality (m=12).


In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_EV} & seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename(columns={'registrations':'ev_registrations'})

ev_registrations.tail(12)

### 1.3 Non EV Demand Forecast
**Procedure**: Using **Auto-ARIMA** to find optimal parameters and fitting a **SARIMAX** model to the **log of Non EV registrations**.
**Datasets Used**: `non_ev_registrations`.
**Main Assumptions**:
- *Assumption*: Log-transformation handles exponential growth better.
- *Rationale*: Registration trends often show multiplicative seasonality.
- *Assumption*: 12-month seasonality (m=12).


In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_nEV_model = pm.auto_arima(np.log(non_ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_nEV = auto_nEV_model.order
best_seasonal_order_nEV = auto_nEV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_nEV} & seasonal_order={best_seasonal_order_nEV} with drift")

nEV_model = sm.tsa.statespace.SARIMAX(
    np.log(non_ev_registrations['registrations']), 
    trend='c' if auto_nEV_model.with_intercept==True else 'n',
    order=best_order_nEV, 
    seasonal_order=best_seasonal_order_nEV
)
nEV_results = nEV_model.fit(disp=False)

forecast_steps = 24
nEV_model_forecast = nEV_results.get_forecast(steps=forecast_steps)

nEV_model_forecast = (nEV_model_forecast.summary_frame(alpha=0.5))
nEV_fc_mean = round(np.exp(nEV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = non_ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': nEV_fc_mean
}, index=forecast_dates)


non_ev_registrations = pd.concat([non_ev_registrations, df_ev_forecast]).rename(columns={'registrations':'non_ev_registrations'})

non_ev_registrations

### 1.2 Data Aggregation
**Procedure**: Grouping and counting registrations by month for both EV and Non-EV categories.


In [ ]:
vehicle_registrations = ev_registrations.merge(non_ev_registrations,how='inner',left_index=True, right_index=True)
vehicle_registrations

**Procedure**: Calculating total registration counts and current EV percentage.


In [ ]:
vehicle_registrations_total = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations["non_ev_registrations"].sum()],
})

vehicle_registrations_total["pct_ev_registrations"] = (
    vehicle_registrations_total["total_ev_registrations"] /
    (vehicle_registrations_total["total_ev_registrations"] + vehicle_registrations_total["total_non_ev_registrations"])
)
vehicle_registrations_total

**Procedure**: Isolating the 2027 forecast period to determine future EV penetration percentage.


In [ ]:
vehicle_registrations_2027 = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["non_ev_registrations"].sum()],
})

vehicle_registrations_2027["pct_ev_registrations"] = (
    vehicle_registrations_2027["total_ev_registrations"] /
    (vehicle_registrations_2027["total_ev_registrations"] + vehicle_registrations_2027["total_non_ev_registrations"])
)
vehicle_registrations_2027

# Phase 2: Spatial Backbone Foundation <a name="phase-2"></a>

We discretize the road network into equidistant points and enrich them with traffic and infrastructure proximity data.


### 2.1 Road Discretization
**Procedure**: Converting LineString road geometries into discrete points every 200 meters using `discretize_backbone_roads`.
**Datasets Used**: `gdf_roads`.
**Main Assumptions**:
- *Assumption*: `sampling_interval_m` = 200.
- *Rationale*: Provides enough resolution for charging station placement without excessive compute.


In [ ]:
params['sampling_interval_m']

In [ ]:
gdf_points = discretize_backbone_roads(
    gdf_roads,
    sampling_interval_m=params['sampling_interval_m']
)
print(f"Generated {len(gdf_points)} points along the backbone.")
gdf_points.head()

### Visualization: Base Point Network
Below we see the result of the discretization. Each red dot represents a sampling point where we will later attach traffic and infrastructure data.

**Visualization**: Displaying the discretized backbone points across Spain.


In [ ]:
m0 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
gdf_plot = gdf_points.to_crs(4326)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='red',
        fill=True
    ).add_to(m0)
m0

### 2.2 Traffic Mapping
**Procedure**: Mapping the closest traffic segments to our backbone points using `map_traffic_to_points`.
**Datasets Used**: `gdf_points`, `gdf_traffic`.
**Main Assumptions**:
- *Assumption*: 
    - `buffer_radius_m` = 50. all traffic route geometries that passes by a 50m radius from the backbone road point is considered as traffic that passes through that backbone road point. This is important since the gdf_traffic has different geometries like the two sides of a highway. 
    - We do not take into account the traffic that passes by the point but has a different direction, like a bridge intersection. To filter out this traffic we ensure that each traffic road geometry has to touch at least 2 consecutive points of the backbone road points. 
- *Rationale*: Captures traffic from the correct road even with slight GPS inaccuracies.


In [ ]:
print(f"buffer_radius_m: {params['buffer_radius_m']}, traffic_columns:{params['traffic_columns']}")

In [ ]:
gdf_points = map_traffic_to_points(
    gdf_points,
    gdf_traffic,
    params['traffic_columns'],
    params['buffer_radius_m']
)
gdf_points[params['traffic_columns']].describe()


### Map 1: Road & Traffic
Points are colored based on their traffic intensity (`total_max`). Green indicates light traffic, yellow mid traffic, and red heavy traffic.

**Visualization**: Backbone points colored by traffic intensity.


In [ ]:
import branca.colormap as cm

m1 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=gdf_points['total_max'].quantile(0.95))
colormap.caption = 'Traffic Intensity (Total Max)'
colormap.add_to(m1)

gdf_plot = gdf_points.to_crs(4326)
for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=colormap(row['total_max']),
        fill=True,
        tooltip=f"Traffic: {row['total_max']}"
    ).add_to(m1)
m1

### 2.3 Existing Infrastructure Proximity
**Procedure**: Calculating the distance to the nearest existing EV charging station for every point.
**Datasets Used**: `gdf_points`, `gdf_chargers`.
**Main Assumptions**:
- *Assumption*: `max_distance` = 100km.  
Any point of the backbone road points is mapped to its nearest EV Charging station, but if the distance to the nearest charging station is above 100km that point is selected as a point that doesn't have a 'near' charging station. This max parameter is created to prevent outliers generation.


In [ ]:
params['max_distance_proximity']

In [ ]:
gdf_points = assign_nearest_charging_stations(
    gdf_points = gdf_points,
    gdf_chargers = gdf_chargers,
    max_distance = params['max_distance_proximity']
)

**Procedure**: Calculating the distance to the nearest traditional gas station (potential conversion sites).
**Datasets Used**: `gdf_points`, `gdf_gas`.
**Main Assumptions**:
- *Assumption*: `max_distance` = 100km.  
Any point of the backbone road points is mapped to its nearest traditional gas station, but if the distance to the nearest traditional gas station is above 100km that point is selected as a point that doesn't have a 'near' traditional gas station. This max parameter is created to prevent outliers generation.


In [ ]:
gdf_points = assign_nearest_gas_stations(
    gdf_points = gdf_points,
    gdf_gas = gdf_gas,
    max_distance = params['max_distance_proximity']
)
print("Proximity analysis completed.")

### Map 2: Road & Current EV Chargers
This map highlights the areas with existing infrastructure. Points are colored by distance to the nearest charger (Green = Close, Red = Far).

**Visualization**: Points colored by their distance to the nearest existing charger (identifying gaps).


In [ ]:
gdf_plot = gdf_points.to_crs(4326)

import branca.colormap as cm

m2 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
dist_colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=max(gdf_plot['dist_charger_m']))
dist_colormap.caption = 'Distance to Nearest Charger (m)'
dist_colormap.add_to(m2)

# Explicitly fill NaN values in 'dist_charger_m' before iteration
gdf_plot['dist_charger_m'] = gdf_plot['dist_charger_m'].fillna(max(gdf_plot['dist_charger_m']))

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=dist_colormap(min(row['dist_charger_m'], 100000)), # Use the cleaned column directly
        fill=True
    ).add_to(m2)
m2

# Phase 3: Grid-Aware Optimization <a name="phase-3"></a>

Solving for the optimal location of new charging sites while considering electrical grid constraints.


### 3.1 Optimization Setup
**Procedure**: Importing the grid-aware optimization engine.


In [ ]:
from scripts.processing.optimize_grid_aware_placement import generate_smart_candidates, solve_grid_aware_optimization, report

**Procedure**: Extracting the 2027 EV penetration percentage for the model.
- *Assumption*: For our optimization model, we assume that out of the total traffic on each road point, only a '2027 EV penetration percentage' would be from Electric Vehicles

In [ ]:
ev_traffic_pct = vehicle_registrations_2027['pct_ev_registrations'].iloc[0]
print(ev_traffic_pct)

**Procedure**: Calculating the ratio of medium-distance traffic that likely needs charging.
- *Assumption*: For our optimization model, we assume that from the total traffic of any point, only the traffic that passes by that point for a medium route segment is the one that needs to recharge at this point. The short route segment traffic don't need to recharge, while the long route segment traffic is not from electric vehicles. 

- *Final Assumption*: The amount of cars that needs to recharge at any specific point i would be 
    - EV_needs_recharge_i = Total_traffic_i  * ev_traffic_pct *  need_charge_pct

In [ ]:
need_charge_pct = gdf_points['medio_max'].sum() / gdf_points['total_max'].sum()
need_charge_pct

**Procedure**: Setting the limit for chargers per site based on existing maximums.
- *Assumption*: For our optimization model, we set that each charging station would have a maximum chargers, limited by the existing max infrastructure of a charging station. This prevents having a charging station with more charging points that the infrastructure could potentially handle (for example a charging station with 200 chargers)   


In [ ]:
max_chargers_per_site =  gdf_chargers['charger_count'].max()
max_chargers_per_site

### 3.2 Optimization Parameters & Assumptions
**Procedure**: Defining the constraints and penalties for the MILP model.
**Main Assumptions**:
- `coverage_threshold_m` (30km): Maximum distance a user should travel to a charger. The EU regulations state that each charging station should be max 60km away from the next charging station, max distance between charging stations should be 60km. To meet this requirement at any specific point of the road the next charging station should be max at 30km (60km/2).  
- `sessions_per_charger` (24): Maximum charging sessions a single charger can handle per day. The average ultra fast sessions from 20%-80% of the batery of a EV could last between 20-40 minutes, plus a possible no usage rate, we calculate that each session would last 1h, so in a day a charger would have 24 sessions. 
- `substation_threshold_m` (10km): Maximum distance to connect a site to a substation. Due to physical restrictions and feasibility, a charging station should be supplied by the nearest electric substation with a max distance of 10km between them. if the distance between the station and the substation is greater than 10km the connection is not possible.
- `power_per_charger_kw` (150kW): Assumed power of new ultra-fast chargers. power demand for each ultra fast charger

- `Optimization penalties (restrictions)`
    - `penalty_coverage`: Cost of leaving a demand point uncovered. 
    - `penalty_supply`: Cost of unmet charging demand.
    - `penalty_grid_upgrade`: Priority for sites that don't require immediate grid upgrades.
    - `penalty_coverage` > `penalty_supply` > `penalty_grid_upgrade`

In [ ]:
coverage_threshold_m = 30000
sessions_per_charger = 24
substation_threshold_m = 10000
power_per_charger_kw = 150
penalty_coverage = 1e6
penalty_supply = 1e4
penalty_grid_upgrade = 1e2

**Procedure**: Extracting the 2027 EV penetration percentage for the model.


In [ ]:
df_cand = generate_smart_candidates(
    gdf_backbone = gdf_points, 
    gdf_chargers = gdf_chargers, 
    gdf_gas = gdf_gas, 
    ev_traffic_pct = ev_traffic_pct, 
    need_charge_pct = need_charge_pct, 
    coverage_threshold_m = coverage_threshold_m, 
    max_chargers_per_site = max_chargers_per_site, 
    sessions_per_charger = sessions_per_charger
    )

**Procedure**: run the optimization process.


In [ ]:
df_res, grid_slacks = solve_grid_aware_optimization(
        gdf_backbone = gdf_points, 
        df_cand = df_cand, 
        gdf_grid = gdf_capacity,
        ev_traffic_pct = ev_traffic_pct,
        need_charge_pct= need_charge_pct,
        coverage_threshold_m = coverage_threshold_m,
        substation_threshold_m = substation_threshold_m,
        max_chargers_per_site = max_chargers_per_site,
        sessions_per_charger = sessions_per_charger,
        power_per_charger_kw = power_per_charger_kw,
        penalty_coverage = penalty_coverage,
        penalty_supply = penalty_supply,
        penalty_grid_upgrade = penalty_grid_upgrade
    )



### 3.3 Results Summary
**Procedure**: Generating a high-level report of the recommended infrastructure roadmap.


In [ ]:
report(df_res, grid_slacks)

In [ ]:
gdf_res = gpd.GeoDataFrame(df_res, geometry='geometry', crs=gdf_points.crs)
gdf_res.to_parquet("data/processed/grid_aware_optimized_sites.parquet")

In [ ]:
gdf_res

# Phase 4: Output Construction <a name="phase-4"></a>

In this final phase, we process the optimization results to generate a clean, actionable dataset. This involves calculating infrastructure requirements, enriching the data with spatial attributes (coordinates and road names), and filtering the results to focus on the newly proposed charging locations.


### 4.1 Capacity Estimation
**Procedure**: Calculating the total estimated power demand (kW) for each site based on the number of chargers.
**Datasets Used**: `gdf_res`.
**Main Assumptions**:
- *Assumption*: Power per charger = 150 kW.
- *Rationale*: We target ultra-fast charging to minimize user wait times and maximize turnover at each site.


In [ ]:
gdf_res['estimated_demand_kw'] = gdf_res['final_n'] * 150

### 4.2 Coordinate Extraction
**Procedure**: Extracting Latitude and Longitude from the point geometries.
**Datasets Used**: `gdf_res`.
**Rationale**: Explicit coordinates are required for integration with GIS tools, mapping APIs, and downstream reporting formats.


In [ ]:
gdf_res['latitude'] = gdf_res.geometry.y
gdf_res['longitude'] = gdf_res.geometry.x

### 4.3 Result Filtering & Investment Isolation
**Procedure**: Filtering the optimization results to isolate sites where infrastructure was successfully placed.
**Datasets Used**: `gdf_res` (the optimization output that has all possible candidates to create a charging station, wether its selected as a EV charging station or not)
**Rationale (The "Why")**:
1. **Operational Focus**: We filter for `is_open == 1` to ignore candidates that the model rejected due to low demand or grid constraints.
2. **New Infrastructure identification**: By further filtering for `type != 'existing'`, we create a specific list of **newly proposed sites** (at gas stations or greenfields). This is critical for budgeting and project management, as it separates maintenance of current sites from new construction initiatives.


In [ ]:
df_stations = gdf_res[(gdf_res['is_open']==1)&(gdf_res['type']!='existing')]

In [ ]:
gdf_res[(gdf_res['is_open']==1)]

### 4.4 Unique Site Identification
**Procedure**: Assigning a unique, human-readable ID (`IBE_XXX`) to each proposed location.
**Rationale**: Standardized IDs simplify tracking, permit identification, and stakeholder reporting.


In [ ]:
df_stations['location_id'] = [f"IBE_{i:03d}" for i in range(1, len(df_stations) + 1)]

### 4.5 Spatial Merge & Road Contextualization
**Procedure**: Performing a **nearest spatial join** (effectively a spatial merge) between our proposed sites and the original road backbone.
**Datasets Used**: `df_stations`, `gdf_points`.
**Rationale (The "Why")**:
1. **Actionable Mapping**: While the model optimizes points in space, stakeholders (engineers, permit managers) need to know which highway each site serves. This "merge" maps coordinates back to human-readable road names (e.g., A-2, AP-7).
2. **Data Enrichment**: This step ensures that the final output contains all necessary metadata from the initial backbone analysis, bridging the gap between raw spatial optimization and infrastructure planning.


In [ ]:
df_stations_indexed = df_stations.set_index('location_id')

df_stations_with_road = gpd.sjoin_nearest(
    df_stations_indexed,
    gdf_points[['road_name', 'geometry']],
    how='left', 
)

df_stations_with_road = df_stations_with_road.loc[~df_stations_with_road.index.duplicated(keep='first')]

df_stations_with_road.rename(columns={'road_name': 'nearest_road_name'}, inplace=True)

df_stations = df_stations.set_index('location_id')
df_stations['route_segment'] = df_stations_with_road['nearest_road_name']
df_stations = df_stations.reset_index() 


In [ ]:
df_stations

### 4.6 Grid Capacity Enrichment
**Procedure**: Merging the proposed charging sites with the electric grid capacity dataset to retrieve technical substation metadata.
**Datasets Used**: `df_stations`, `gdf_capacity`.
**Rationale**: To evaluate the feasibility of our infrastructure roadmap, we must associate every proposed site with the actual electrical limits of its connecting substation. This **merge** links each site's `substation_id` to its available `capacity_kw` and operating `company`.


In [ ]:
capacity_usefull_cols = gdf_capacity[['company','row_id','capacity_kw']]

df_stations = df_stations.merge(
    capacity_usefull_cols,
    left_on = 'substation_id',
    right_on = 'row_id',
    how = 'left'
    )

df_stations.head()

### 4.7 Grid Stress Analysis & Feasibility Status
**Procedure**: Aggregating the total proposed demand at the substation level and calculating a **Capacity-to-Demand Ratio** to determine grid status.
**Datasets Used**: `df_stations`.
**Main Assumptions (Logic)**:
- **Ratio Calculation**: `capacity_kw / estimated_demand_kw`.
- **Status Classification**:
    - **Congested**: Ratio > 1.2 or 0 (High demand relative to capacity, 20% more than the available capacity, or 0 capacity).
    - **Moderate**: Ratio > 1 (Demand is higher than the available capacity but lower than a 20% more than the available capacity).
    - **Sufficient**: Ratio <= 1 (Capacity exceeds demand).
**Rationale**: This analysis identifies potential grid bottlenecks. By calculating the ratio of available capacity to the new infrastructure load, we can prioritize sites that are "grid-ready" and flag those requiring significant reinforcement.


In [ ]:
subs_demand = df_stations.groupby('substation_id')[['estimated_demand_kw','capacity_kw']].agg({'estimated_demand_kw':'sum','capacity_kw':'max'}).reset_index()

subs_demand['ratio'] = subs_demand['capacity_kw']/subs_demand['estimated_demand_kw']
subs_demand['grid_status'] = np.where(
    (subs_demand['ratio'] >1.2)|(subs_demand['ratio']==0), 'Congested',
    np.where(subs_demand['ratio']>1, 'Moderate', 'Sufficient')
)

df_final = df_stations.merge(
    subs_demand[['substation_id','ratio','grid_status']],
    on = 'substation_id',
    how = 'left'
    )

In [ ]:
file_2 = df_final[['location_id','latitude','longitude','route_segment','final_n','grid_status']]
file_2.rename(columns = {'final_n': 'n_chargers_proposed'},inplace = True)
file_2.head()

In [ ]:
file_2.to_csv('data/outputs/File 2.csv')

In [ ]:
file_3 = df_final[['latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3 = file_3[file_3['grid_status']!= 'Sufficent']
file_3['bottleneck_id'] = [f"FRIC_{i:03d}" for i in range(1, len(file_3) + 1)]
file_3 = file_3[['bottleneck_id','latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3.rename(columns = {'company': 'distributor_network'}, inplace = True )
file_3.head()

In [ ]:
file_3.to_csv('data/outputs/File 3.csv')

In [ ]:
file_1 = pd.DataFrame({
    'total_proposed_stations': [len(file_2)],
    'total_exisiting_stations_baseline': [len(gdf_chargers)],
    'total_friction_points': [len(file_3)],
    'total_ev_projected_2027' :  [vehicle_registrations_total['total_ev_registrations'].item()]
})
file_1

In [ ]:
file_1.to_csv('data/outputs/File 1.csv')

In [ ]:

# 1. Map Colors
color_map = {
    'Sufficient': 'green',
    'Moderate': 'yellow',
    'Congested': 'red'
}

# 2. Coordinate Transformation (Optional: only if coordinates are still in projected meters)
# Assuming source is EPSG:3042 (Spain UTM) and converting to WGS84 for Folium
if 'geometry' in file_2.columns and isinstance(file_2, gpd.GeoDataFrame):
    df_map = file_2.to_crs(epsg=4326)
else:
    # If it's a CSV with X/Y columns in meters, create GeoDataFrame first
    gdf = gpd.GeoDataFrame(
        file_2, 
        geometry=gpd.points_from_xy(file_2.longitude, file_2.latitude),
        crs="EPSG:3042"
    )
    df_map = gdf.to_crs(epsg=4326)

# 3. Initialize Map
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')

# 4. Add Points to Map
for _, row in df_map.iterrows():
    # Use Tooltip for hover information
    tooltip_html = f"""
        <b>Route Segment:</b> {row['route_segment']}<br>
        <b>Proposed Chargers:</b> {row['n_chargers_proposed']}<br>
        <b>Grid Status:</b> {row['grid_status']}<br>
        <b>Coordinates:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})
    """
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color_map.get(row['grid_status'], 'gray'),
        fill=True,
        fill_color=color_map.get(row['grid_status'], 'gray'),
        fill_opacity=0.7,
        tooltip=tooltip_html  # Shows on hover
    ).add_to(m)

# Display Map
m.save("grid_status_map.html") # Or just 'm' in a notebook

m


